In [1]:
!pip install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.1 MB/s eta 0:00:00a 0:00:01


In [2]:
from Bio import SeqIO

def load_sequences(fasta_path):
    sequences = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        pid = record.id   # keep entire ID (safe)
        sequences[pid] = str(record.seq)
    return sequences

train_sequences = load_sequences("/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta")

print("Total sequences loaded:", len(train_sequences))


Total sequences loaded: 82404


In [3]:
import pandas as pd

def load_go_terms(tsv_path):
    df = pd.read_csv(tsv_path, sep="\t", header=None, names=["protein_id", "go_term"])
    
    # Group GO terms for each protein
    grouped = df.groupby("protein_id")["go_term"].apply(list).reset_index()
    
    return grouped

train_terms = load_go_terms("/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv")

print("Total proteins with GO labels:", len(train_terms))
train_terms.head()


Total proteins with GO labels: 26126


,protein_id,go_term
0,GO:0000001,"[P, P, P, P, P, P, P, P, P, P, P, P, P, P, P, ..."
1,GO:0000002,"[P, P, P, P, P, P, P, P, P, P, P, P, P, P, P, ..."
2,GO:0000006,"[F, F, F]"
3,GO:0000007,[F]
4,GO:0000009,"[F, F, F, F, F, F, F, F, F, F, F, F]"


In [4]:
import pandas as pd

# Load all three columns
terms_df = pd.read_csv(
    "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv",
    sep="\t",
    header=None,
    names=["protein_id", "go_term", "ontology"]
)

# Group GO terms for each protein
protein_to_terms = terms_df.groupby("protein_id")["go_term"].apply(list).reset_index()

print("Total proteins with GO labels:", len(protein_to_terms))
protein_to_terms.head()


Total proteins with GO labels: 82405


,protein_id,go_term
0,A0A023FBW4,[GO:0019958]
1,A0A023FBW7,[GO:0019957]
2,A0A023FDY8,[GO:0019957]
3,A0A023FF81,[GO:0019958]
4,A0A023FFB5,[GO:0019957]


In [5]:
import pandas as pd

# Convert train_sequences dict to dataframe
seq_df = pd.DataFrame(list(train_sequences.items()), columns=["protein_id", "sequence"])

# Merge with GO terms
train_df = seq_df.merge(protein_to_terms, on="protein_id", how="inner")

print("Final training dataset shape:", train_df.shape)
train_df.head()


Final training dataset shape: (0, 3)


,protein_id,sequence,go_term


In [6]:
from Bio import SeqIO

def load_sequences_correct(fasta_path):
    sequences = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        # Extract UniProt accession (middle part of header)
        if "|" in record.id:
            pid = record.id.split("|")[1]
        else:
            pid = record.id
        sequences[pid] = str(record.seq)
    return sequences

train_sequences = load_sequences_correct(
    "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta"
)

print("Total sequences loaded:", len(train_sequences))
list(train_sequences.keys())[:5]


Total sequences loaded: 82404


['A0A0C5B5G6', 'A0JNW5', 'A0JP26', 'A0PK11', 'A1A4S6']

In [7]:
import pandas as pd

seq_df = pd.DataFrame(list(train_sequences.items()), columns=["protein_id", "sequence"])

train_df = seq_df.merge(protein_to_terms, on="protein_id", how="inner")

print("Final training dataset shape:", train_df.shape)
train_df.head()


Final training dataset shape: (82404, 3)


,protein_id,sequence,go_term
0,A0A0C5B5G6,MRWQEMGYIFYPRKLR,"[GO:0001649, GO:0033687, GO:0005615, GO:000563..."
1,A0JNW5,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,"[GO:0120013, GO:0034498, GO:0005769, GO:012000..."
2,A0JP26,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,[GO:0005515]
3,A0PK11,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,"[GO:0007605, GO:0005515]"
4,A1A4S6,MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...,"[GO:0005829, GO:0010008, GO:0005515, GO:000509..."


In [8]:
!pip install obonet networkx


In [9]:
import obonet
import networkx as nx

go_graph = obonet.read_obo(
    "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
)
print("Total GO terms in graph:", len(go_graph))


Total GO terms in graph: 40122


In [10]:
def get_ancestors(go_id, graph):
    """
    Returns all ancestor GO terms for a given GO term in the DAG.
    """
    if go_id not in graph:
        return set()
    return nx.ancestors(graph, go_id)


In [11]:
def propagate_go_terms(go_terms, graph):
    """
    Takes a list of GO terms and returns the list including all ancestors.
    """
    propagated = set(go_terms)
    for go in go_terms:
        propagated.update(get_ancestors(go, graph))
    return list(propagated)


In [12]:
train_df['go_term_propagated'] = train_df['go_term'].apply(lambda x: propagate_go_terms(x, go_graph))

# Check a sample
train_df[['protein_id', 'go_term', 'go_term_propagated']].head()


,protein_id,go_term,go_term_propagated
0,A0A0C5B5G6,"[GO:0001649, GO:0033687, GO:0005615, GO:000563...","[GO:1990955, GO:0005689, GO:0046103, GO:000617..."
1,A0JNW5,"[GO:0120013, GO:0034498, GO:0005769, GO:012000...","[GO:0031868, GO:0002020, GO:0043183, GO:003418..."
2,A0JP26,[GO:0005515],"[GO:0031868, GO:0002020, GO:0043183, GO:003418..."
3,A0PK11,"[GO:0007605, GO:0005515]","[GO:0031868, GO:0002020, GO:0043183, GO:003418..."
4,A1A4S6,"[GO:0005829, GO:0010008, GO:0005515, GO:000509...","[GO:0031868, GO:0002020, GO:0043183, GO:003418..."


In [13]:
# 20 standard amino acids + unknown
AA_DICT = {aa: i+1 for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
AA_DICT["X"] = 21  # unknown / padding

def sequence_to_indices(seq, max_len=1024):
    """
    Convert sequence to list of indices (padded to max_len)
    """
    seq = seq[:max_len]
    indices = [AA_DICT.get(aa, 21) for aa in seq]
    # pad if shorter
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    return indices


In [14]:
MAX_LEN = 1024
train_df['seq_indices'] = train_df['sequence'].apply(lambda x: sequence_to_indices(x, max_len=MAX_LEN))

# Check
train_df[['protein_id', 'sequence', 'seq_indices']].head()


,protein_id,sequence,seq_indices
0,A0A0C5B5G6,MRWQEMGYIFYPRKLR,"[11, 15, 19, 14, 4, 11, 6, 20, 8, 5, 20, 13, 1..."
1,A0JNW5,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,"[11, 1, 6, 8, 8, 9, 9, 14, 8, 10, 9, 7, 10, 16..."
2,A0JP26,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,"[11, 18, 1, 4, 18, 2, 16, 11, 13, 1, 1, 16, 1,..."
3,A0PK11,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,"[11, 13, 6, 19, 5, 9, 9, 1, 19, 20, 6, 10, 1, ..."
4,A1A4S6,MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...,"[11, 6, 10, 14, 13, 10, 4, 5, 16, 3, 2, 20, 10..."


In [15]:
# Get all unique GO terms in training set
all_go_terms = sorted({go for gos in train_df['go_term_propagated'] for go in gos})
go2idx = {go: i for i, go in enumerate(all_go_terms)}
idx2go = {i: go for go, i in go2idx.items()}

NUM_CLASSES = len(all_go_terms)
print("Number of unique GO terms (classes):", NUM_CLASSES)


Number of unique GO terms (classes): 39791


In [16]:
import numpy as np

def go_to_multi_hot(go_list, go2idx):
    vec = np.zeros(len(go2idx), dtype=np.float32)
    for go in go_list:
        if go in go2idx:
            vec[go2idx[go]] = 1.0
    return vec


In [17]:
!pip install fair-esm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
from tqdm import tqdm
import esm
import pandas as pd
import numpy as np

# --- Configuration ---
OUTPUT_DIR = "esm_embeddings_t33_650M"
BATCH_SIZE = 1  # 650M is heavy; stick to 1 for stability or small batches if GPU VRAM > 16GB
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. Load ESM-2 650M (33 Layers) ---
# Model name: esm2_t33_650M_UR50D
esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
esm_model = esm_model.half().to(device) # Use half precision to save memory
esm_model.eval()
batch_converter = alphabet.get_batch_converter()
ESM_DIM = esm_model.embed_dim  # This will be 1280
print(f"✅ ESM-2 650M Loaded. Embedding Dim: {ESM_DIM}")

# --- 2. Dataset Logic ---
class ProteinDataset(Dataset):
    def __init__(self, df):
        self.sequences = df["sequence"].values
        self.labels = df["go_term_propagated"].values 

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def collate_fn(batch):
    sequences, targets = zip(*batch)
    return list(sequences), list(targets)

# Initialize DataLoader
try:
    dataset = ProteinDataset(train_df)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
except NameError:
    print("❌ Error: 'train_df' not found. Ensure previous cells were run.")

# --- 3. Chunked Embedding Generation ---
chunk_size = 512 # Max residues per chunk to prevent OOM
repr_layer = 33  # The last layer for 650M model

print(f"Starting embedding generation for {len(dataset)} proteins...")

with torch.no_grad():
    for batch_idx, (sequences_batch, targets_batch) in enumerate(tqdm(loader)):
        for protein_idx, seq in enumerate(sequences_batch):
            i = batch_idx * BATCH_SIZE + protein_idx
            target_indices_list = targets_batch[protein_idx]
            all_chunks = []

            # Split long sequences into manageable chunks
            for start in range(0, len(seq), chunk_size):
                sub_seq = seq[start:start+chunk_size]
                _, _, tokens = batch_converter([("protein", sub_seq)])
                tokens = tokens.to(device)

                # Forward pass - extracting from layer 33
                out = esm_model(tokens, repr_layers=[repr_layer], return_contacts=False)
                reps = out["representations"][repr_layer]
                
                # Mean pool the chunk and move to CPU
                pooled = reps.mean(dim=1).squeeze(0).cpu()
                all_chunks.append(pooled.float())

                del tokens, out, reps
                torch.cuda.empty_cache()

            # Final Embedding: Average of all chunks
            final_emb = torch.stack(all_chunks, dim=0).mean(dim=0)

            # Save to disk
            torch.save({
                "embedding": final_emb,
                "targets": target_indices_list 
            }, f"{OUTPUT_DIR}/sample_{i}.pt")

print(f"✅ Generation Complete. Saved to {OUTPUT_DIR}")

In [ ]:
# !zip -r -q [output_zip_name].zip [folder_to_zip]
!zip -r -q esm_embeddings_650M.zip esm_embeddings_t33_650M
print("✅ Zipping complete!")

In [17]:
import pandas as pd
import networkx as nx
import obonet
from Bio import SeqIO
from tqdm.auto import tqdm
import json
import os

# --- 1. REBUILD VOCABULARY (Fixing the NameError) ---
print("🔄 Rebuilding GO_ID_TO_INDEX from source files...")

# Paths
FASTA_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta"
TERMS_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv"
OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"

# A. Get Valid Protein IDs
valid_proteins = set()
for record in SeqIO.parse(FASTA_PATH, "fasta"):
    pid = record.id.split("|")[1] if "|" in record.id else record.id
    valid_proteins.add(pid)

# B. Load Terms & Filter
# Only keep terms for proteins that exist in the sequence file
terms_df = pd.read_csv(TERMS_PATH, sep="\t", header=None, names=["pid", "term", "aspect"])
filtered_terms = terms_df[terms_df['pid'].isin(valid_proteins)]
raw_terms = set(filtered_terms['term'].unique())

# C. Load Graph & Expand Ancestors
go_graph = obonet.read_obo(OBO_PATH)
all_terms = raw_terms.copy()
for term in tqdm(raw_terms, desc="Expanding Ancestors"):
    if term in go_graph:
        all_terms.update(nx.ancestors(go_graph, term))

# D. Create the Dictionary
all_go_terms = sorted(list(all_terms))
GO_ID_TO_INDEX = {go: i for i, go in enumerate(all_go_terms)}

print(f"✅ Vocabulary Rebuilt. Size: {len(GO_ID_TO_INDEX)}")

# --- 2. RUN YOUR JSON GENERATION CODE ---
print(f"Generating ontology map for indices 0-{len(GO_ID_TO_INDEX)-1}...")

OUTPUT_JSON_PATH = "/kaggle/working/go_ontology_map_650M.json"
namespace_map = {'biological_process': 'BP', 'molecular_function': 'MF', 'cellular_component': 'CC'}
ontology_mapping = {}

for go_id, idx in GO_ID_TO_INDEX.items():
    if go_id in go_graph.nodes:
        raw_ns = go_graph.nodes[go_id].get('namespace', 'unknown')
        ontology_mapping[str(idx)] = namespace_map.get(raw_ns, 'unknown')
    else:
        ontology_mapping[str(idx)] = 'unknown'

with open(OUTPUT_JSON_PATH, 'w') as f:
    json.dump(ontology_mapping, f)

print(f"✅ SUCCESS: Map saved to {OUTPUT_JSON_PATH}")

🔄 Rebuilding GO_ID_TO_INDEX from source files...


Expanding Ancestors:   0%|          | 0/26125 [00:00<?, ?it/s]

✅ Vocabulary Rebuilt. Size: 39791
Generating ontology map for indices 0-39790...
✅ SUCCESS: Map saved to /kaggle/working/go_ontology_map_650M.json


In [18]:
import json

# --- Configuration ---
# This matches your new 650M project name
OUTPUT_JSON_PATH = "/kaggle/working/go_ontology_map_650M.json"

# Namespace mapping for CAFA
namespace_map = {
    'biological_process': 'BP',
    'molecular_function': 'MF',
    'cellular_component': 'CC'
}

print(f"Generating ontology map for indices 0-{len(GO_ID_TO_INDEX)-1}...")

ontology_mapping = {}

# Use the variables already in your memory (globals)
for go_id, idx in GO_ID_TO_INDEX.items():
    if go_id in go_graph.nodes:
        # Get the namespace from the OBO graph
        raw_ns = go_graph.nodes[go_id].get('namespace', 'unknown')
        ontology_mapping[str(idx)] = namespace_map.get(raw_ns, 'unknown')
    else:
        ontology_mapping[str(idx)] = 'unknown'

# Save to JSON
with open(OUTPUT_JSON_PATH, 'w') as f:
    json.dump(ontology_mapping, f)

print(f"✅ SUCCESS: New ontology map saved to: {OUTPUT_JSON_PATH}")
print(f"Mapped {len(ontology_mapping)} terms.")

# --- Verify one entry ---
sample_idx = "0"
print(f"Sample mapping: Index {sample_idx} -> {ontology_mapping[sample_idx]}")

Generating ontology map for indices 0-39790...
✅ SUCCESS: New ontology map saved to: /kaggle/working/go_ontology_map_650M.json
Mapped 39791 terms.
Sample mapping: Index 0 -> BP


In [19]:
import obonet
import networkx as nx

# --- 1. Load the OBO Graph ---
OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
print("Loading Gene Ontology graph from OBO file...")
go_graph = obonet.read_obo(OBO_PATH)

# --- 2. Filter and Map Relations ---
# We focus on 'is_a' and 'part_of' as they define the standard CAFA functional constraints
REAL_HIERARCHY_RELATIONS = []

print("Mapping OBO relations to vocabulary indices...")
for child_id, parent_id, key in go_graph.edges(keys=True):
    if key in ['is_a', 'part_of']:
        # Only keep the relation if both terms are in our trained vocabulary
        if child_id in GO_ID_TO_INDEX and parent_id in GO_ID_TO_INDEX:
            child_idx = GO_ID_TO_INDEX[child_id]
            parent_idx = GO_ID_TO_INDEX[parent_id]
            REAL_HIERARCHY_RELATIONS.append((child_idx, parent_idx))

# Remove duplicates to keep the loss calculation efficient
REAL_HIERARCHY_RELATIONS = list(set(REAL_HIERARCHY_RELATIONS))

print(f"✅ Success: Extracted {len(REAL_HIERARCHY_RELATIONS):,} biological relations.")

Loading Gene Ontology graph from OBO file...
Mapping OBO relations to vocabulary indices...
✅ Success: Extracted 66,673 biological relations.


In [20]:
max_idx = max([max(r) for r in REAL_HIERARCHY_RELATIONS])
print(f"Max index in relations: {max_idx} (Model capacity: 39791)")

Max index in relations: 39790 (Model capacity: 39791)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm
import os
import glob
import numpy as np
from sklearn.metrics import f1_score

# --- 1. Hierarchy-Aware Loss Class ---
class HierarchyAwareLoss(nn.Module):
    def __init__(self, hierarchy_relations, lambd=0.3):
        super().__init__()
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.lambd = lambd
        
        # Extract child and parent indices from the relations list
        if hierarchy_relations:
            child_idx = torch.tensor([r[0] for r in hierarchy_relations], dtype=torch.long)
            parent_idx = torch.tensor([r[1] for r in hierarchy_relations], dtype=torch.long)
        else:
            child_idx = torch.empty(0, dtype=torch.long)
            parent_idx = torch.empty(0, dtype=torch.long)
            
        self.register_buffer('child_indices', child_idx)
        self.register_buffer('parent_indices', parent_idx)

    def forward(self, outputs, targets):
        bce = self.bce_loss(outputs, targets)
        
        if self.child_indices.numel() > 0:
            child_probs = torch.sigmoid(outputs[:, self.child_indices])
            parent_probs = torch.sigmoid(outputs[:, self.parent_indices])
            violation = torch.clamp(child_probs - parent_probs, min=0)
            hrc_loss = torch.mean(violation * targets[:, self.child_indices])
            return bce + (self.lambd * hrc_loss)
        
        return bce

# --- 2. UPDATED Dataset Class with Label Mapping ---
class PTEmbeddingDataset(Dataset):
    def __init__(self, pt_dir, label_to_idx, num_classes=39791):
        """
        label_to_idx: Dictionary mapping 'GO:0001234' -> 15 (integer index)
        """
        self.pt_files = sorted(glob.glob(os.path.join(pt_dir, "*.pt")))
        self.label_to_idx = label_to_idx 
        self.num_classes = num_classes
        
        if len(self.pt_files) == 0:
            raise RuntimeError(f"No .pt files found in {pt_dir}")

    def __len__(self):
        return len(self.pt_files)

    def __getitem__(self, idx):
        try:
            data = torch.load(self.pt_files[idx], weights_only=True)
            embedding = data["embedding"].squeeze().float()
            
            # Create target vector
            target_tensor = torch.zeros(self.num_classes, dtype=torch.float32)
            raw_labels = data.get("targets", [])
            
            # Flatten list if nested (e.g. [['GO:001'], ['GO:002']])
            if isinstance(raw_labels, list) and len(raw_labels) > 0 and isinstance(raw_labels[0], list):
                raw_labels = [item for sublist in raw_labels for item in sublist]
                
            # Map String IDs (GO:XXXX) to Integer Indices using the dictionary
            valid_indices = []
            for label in raw_labels:
                if label in self.label_to_idx:
                    valid_indices.append(self.label_to_idx[label])
            
            # Create Multi-hot Vector from valid indices
            if valid_indices:
                valid_indices = list(set(valid_indices)) # Remove duplicates
                target_tensor[torch.tensor(valid_indices, dtype=torch.long)] = 1.0
                
            return embedding, target_tensor
            
        except Exception as e:
            return torch.zeros(1280), torch.zeros(self.num_classes)

# --- 3. Model Definition --- MLP (Multi-Layer Perceptron)  Feed-Forward Neural Network (FFNN) two-layer Multi-Layer Perceptron (MLP)
class EmbeddingClassifier(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x: torch.Tensor):
        return self.model(x)

# --- 4. Initialization & Setup ---
PT_DIR = "/kaggle/input/protien650esm/esm_embeddings_650M/esm_embeddings_t33_650M"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensure GO_ID_TO_INDEX exists
if 'GO_ID_TO_INDEX' not in locals():
    print("⚠️ Warning: GO_ID_TO_INDEX not found. Creating a temporary dummy map for testing (DON'T USE FOR REAL TRAINING).")
    # In real training, this line shouldn't run if you ran the OBO block
    GO_ID_TO_INDEX = {} 

# Initialize Dataset with the mapping dictionary
full_dataset = PTEmbeddingDataset(PT_DIR, label_to_idx=GO_ID_TO_INDEX, num_classes=39791)
print(f"✅ Loaded Dataset with {len(full_dataset)} samples.")

# Data Splitting
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# Loaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

# Model & Loss
model = EmbeddingClassifier(input_dim=1280, num_classes=39791).to(device)

# Ensure REAL_HIERARCHY_RELATIONS exists
if 'REAL_HIERARCHY_RELATIONS' not in locals():
    print("⚠️ Warning: REAL_HIERARCHY_RELATIONS not found. Using empty list.")
    REAL_HIERARCHY_RELATIONS = []

criterion = HierarchyAwareLoss(REAL_HIERARCHY_RELATIONS, lambd=0.3).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# --- 5. Training Loop ---
epochs = 40
best_val_score = 0.0 

print(f"Starting training on {device}...")

for epoch in range(epochs):
    model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    
    for embs, lbls in pbar:
        embs, lbls = embs.to(device), lbls.to(device)
        
        optimizer.zero_grad()
        outputs = model(embs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # Validation
    model.eval()
    val_loss = 0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for embs, lbls in tqdm(val_loader, desc=f"Val"):
            embs, lbls = embs.to(device), lbls.to(device)
            outputs = model(embs)
            
            val_loss += criterion(outputs, lbls).item()
            
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.3).float()
            
            all_preds.append(preds.cpu().numpy())
            all_targets.append(lbls.cpu().numpy())

    # Metrics
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    # Check if we have predictions before vstacking (to avoid crashes on empty vals)
    if len(all_preds) > 0:
        np_preds = np.vstack(all_preds)
        np_targets = np.vstack(all_targets)
        val_f1 = f1_score(np_targets, np_preds, average='micro')
    else:
        val_f1 = 0.0
    
    print(f"\n✨ Epoch {epoch+1} Summary:")
    print(f"   Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"   Val F1-Score: {val_f1:.4f}")

    scheduler.step(avg_val_loss)
    
    if val_f1 > best_val_score:
        best_val_score = val_f1
        torch.save(model.state_dict(), "best_model_hierarchy.pth")
        print("   ⭐ New Best Model Saved!")

print("\n✅ Training Complete.")

In [22]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
import numpy as np
import pandas as pd
import networkx as nx
import obonet
from Bio import SeqIO
from tqdm.auto import tqdm
import glob
import os
import gc

# --- 1. CONFIGURATION ---
print("⚙️ Setting up paths and device...")
OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
TRAIN_TERMS_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv"
TRAIN_FASTA_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta"
PT_DIR = "/kaggle/input/protien650esm/esm_embeddings_650M/esm_embeddings_t33_650M" # Verify this path matches your data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. REBUILD VOCABULARY ---
print("🔄 Rebuilding Vocabulary (GO_ID_TO_INDEX)...")
# We must replicate the exact vocabulary used during training
valid_proteins = set()
for record in SeqIO.parse(TRAIN_FASTA_PATH, "fasta"):
    pid = record.id.split("|")[1] if "|" in record.id else record.id
    valid_proteins.add(pid)

# Load and filter terms
terms_df = pd.read_csv(TRAIN_TERMS_PATH, sep="\t", header=None, names=["pid", "term", "aspect"])
filtered_terms = terms_df[terms_df['pid'].isin(valid_proteins)]
raw_terms = set(filtered_terms['term'].unique())

# Load Graph and Expand Ancestors
go_graph = obonet.read_obo(OBO_PATH)
all_terms = raw_terms.copy()
print("   Expanding ancestors from OBO graph...")
for term in tqdm(raw_terms):
    if term in go_graph: all_terms.update(nx.ancestors(go_graph, term))

# Create Index Map (Sorted to match model output)
all_go_terms = sorted(list(all_terms))
GO_ID_TO_INDEX = {go: i for i, go in enumerate(all_go_terms)}
VOCAB_SIZE = len(GO_ID_TO_INDEX)
print(f"✅ Vocabulary Size: {VOCAB_SIZE}")

# --- 3. PREPARE EVALUATION ARTIFACTS ---
print("🗺️ Building Evaluation Maps & Weights...")
masks = {
    "BP": np.zeros(VOCAB_SIZE, dtype=bool),
    "MF": np.zeros(VOCAB_SIZE, dtype=bool),
    "CC": np.zeros(VOCAB_SIZE, dtype=bool)
}
weights = np.zeros(VOCAB_SIZE, dtype=np.float32)
ancestor_map = {}

# Get term counts for IA (Information Accretion) Weights
term_counts = filtered_terms['term'].value_counts().to_dict()
total_proteins = filtered_terms['pid'].nunique()

for term, idx in tqdm(GO_ID_TO_INDEX.items()):
    # 1. IA Weights: -log2(P(t))
    freq = (term_counts.get(term, 0) + 1) / (total_proteins + 1)
    weights[idx] = -np.log2(freq)
    
    if term in go_graph:
        # 2. Ancestor Map (for Propagation)
        ancs = {GO_ID_TO_INDEX[a] for a in nx.ancestors(go_graph, term) if a in GO_ID_TO_INDEX}
        if ancs: ancestor_map[idx] = ancs
        
        # 3. Ontology Masks
        ns = go_graph.nodes[term].get('namespace')
        if ns == 'biological_process': masks["BP"][idx] = True
        elif ns == 'molecular_function': masks["MF"][idx] = True
        elif ns == 'cellular_component': masks["CC"][idx] = True

# --- 4. DATASET & MODEL CLASS ---
class PTEmbeddingDataset(Dataset):
    def __init__(self, pt_dir, label_to_idx, num_classes):
        self.pt_files = sorted(glob.glob(os.path.join(pt_dir, "*.pt")))
        self.label_to_idx = label_to_idx
        self.num_classes = num_classes
    def __len__(self): return len(self.pt_files)
    def __getitem__(self, idx):
        data = torch.load(self.pt_files[idx], weights_only=True)
        emb = data["embedding"].float()
        target_tensor = torch.zeros(self.num_classes, dtype=torch.float32)
        
        # Handle labels (ensure flat list of strings)
        raw_labels = data.get("targets", [])
        if isinstance(raw_labels, list) and len(raw_labels)>0 and isinstance(raw_labels[0], list):
            raw_labels = [item for sublist in raw_labels for item in sublist]
            
        indices = [self.label_to_idx[t] for t in raw_labels if t in self.label_to_idx]
        if indices: target_tensor[list(set(indices))] = 1.0
        return emb, target_tensor

class EmbeddingClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.model(x)

# Setup Data (90% Train / 10% Validation)
full_ds = PTEmbeddingDataset(PT_DIR, GO_ID_TO_INDEX, VOCAB_SIZE)
train_len = int(0.9 * len(full_ds))
val_len = len(full_ds) - train_len
_, val_ds = random_split(full_ds, [train_len, val_len], generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

# --- 5. LOAD BEST MODEL ---
print("\n⚙️ Loading Model...")
model = EmbeddingClassifier(1280, VOCAB_SIZE).to(device)
model_path = "/kaggle/input/protien650esm/best_model_hierarchy.pth"

if os.path.exists(model_path):
    print(f"✅ Loading BEST saved weights from: {model_path}")
    model.load_state_dict(torch.load(model_path, map_location=device))
else:
    print("⚠️ WARNING: 'best_model_hierarchy.pth' not found. Using random initialized weights.")

# --- 6. RUN INFERENCE ---
print("\n🚀 Running Inference on Validation Set...")
model.eval()
val_probs = []
val_targets = []

with torch.no_grad():
    for embs, targets in tqdm(val_loader, desc="Inference"):
        embs = embs.to(device)
        out = model(embs)
        val_probs.append(torch.sigmoid(out).cpu().numpy())
        val_targets.append(targets.numpy())

y_prob_raw = np.vstack(val_probs)
y_true = np.vstack(val_targets)
print(f"   Inference Done. Predictions Shape: {y_prob_raw.shape}")

# Clean up memory
del model, val_loader, full_ds, val_ds
gc.collect()

# --- 7. PROPAGATE & EVALUATE ---
print("\n🌊 Propagating Predictions (Enforcing Hierarchy)...")
# In-place propagation: Parent Probability = max(Parent, Child)
for child, parents in tqdm(ancestor_map.items()):
    if parents:
        np.maximum(y_prob_raw[:, list(parents)], y_prob_raw[:, child][:, None], out=y_prob_raw[:, list(parents)])

print("\n📊 Calculating Weighted Metrics...")
def calculate_metrics(yt, yp, w, thresholds, mask=None):
    if mask is not None:
        yt, yp, w = yt[:, mask], yp[:, mask], w[mask]
    
    best = {"F1": 0.0}
    true_sums = (yt * w).sum(1)
    
    for t in thresholds:
        pred = (yp >= t).astype(int)
        tp = ((pred * yt) * w).sum(1)
        pred_sum = (pred * w).sum(1)
        
        with np.errstate(divide='ignore', invalid='ignore'):
            p = np.divide(tp, pred_sum, out=np.zeros_like(pred_sum), where=pred_sum!=0).mean()
            r = np.divide(tp, true_sums, out=np.zeros_like(true_sums), where=true_sums!=0).mean()
        
        f1 = (2*p*r)/(p+r) if (p+r)>0 else 0
        if f1 > best["F1"]: 
            best = {"F1": f1, "Precision": p, "Recall": r, "Threshold": t}
    return best

results = []
thresholds = np.arange(0.1, 0.9, 0.05)

for name, mask in {"Global": None, "BP": masks["BP"], "MF": masks["MF"], "CC": masks["CC"]}.items():
    print(f"   Evaluating {name}...")
    m = calculate_metrics(y_true, y_prob_raw, weights, thresholds, mask)
    results.append({
        "Ontology": name,
        "Weighted F1": f"{m['F1']:.4f}",
        "Precision": f"{m['Precision']:.4f}",
        "Recall": f"{m['Recall']:.4f}",
        "Best Threshold": f"{m['Threshold']:.2f}"
    })

print("\n🏆 FINAL RESULTS 🏆")
print(pd.DataFrame(results).to_markdown(index=False))

⚙️ Setting up paths and device...
🔄 Rebuilding Vocabulary (GO_ID_TO_INDEX)...
   Expanding ancestors from OBO graph...


  0%|          | 0/26125 [00:00<?, ?it/s]

✅ Vocabulary Size: 39791
🗺️ Building Evaluation Maps & Weights...


  0%|          | 0/39791 [00:00<?, ?it/s]


⚙️ Loading Model...
✅ Loading BEST saved weights from: /kaggle/input/protien650esm/best_model_hierarchy.pth

🚀 Running Inference on Validation Set...


Inference:   0%|          | 0/129 [00:00<?, ?it/s]

   Inference Done. Predictions Shape: (8241, 39791)

🌊 Propagating Predictions (Enforcing Hierarchy)...


  0%|          | 0/18006 [00:00<?, ?it/s]


📊 Calculating Weighted Metrics...
   Evaluating Global...
   Evaluating BP...
   Evaluating MF...
   Evaluating CC...

🏆 FINAL RESULTS 🏆
| Ontology   |   Weighted F1 |   Precision |   Recall |   Best Threshold |
|:-----------|--------------:|------------:|---------:|-----------------:|
| Global     |        0.4838 |      0.4338 |   0.5468 |             0.25 |
| BP         |        0.1442 |      0.1232 |   0.1737 |             0.1  |
| MF         |        0.4561 |      0.4045 |   0.5229 |             0.1  |
| CC         |        0.4237 |      0.3779 |   0.4822 |             0.25 |


In [21]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
import numpy as np
import pandas as pd
import networkx as nx
import obonet
from Bio import SeqIO
from tqdm.auto import tqdm
import glob
import os
import gc

# --- 1. CONFIGURATION ---
print("⚙️ Setting up paths and device...")
OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
TRAIN_TERMS_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv"
TRAIN_FASTA_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta"
PT_DIR = "/kaggle/input/protien650esm/esm_embeddings_650M/esm_embeddings_t33_650M" # Verify this path matches your data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. REBUILD VOCABULARY ---
print("🔄 Rebuilding Vocabulary (GO_ID_TO_INDEX)...")
# We must replicate the exact vocabulary used during training
valid_proteins = set()
for record in SeqIO.parse(TRAIN_FASTA_PATH, "fasta"):
    pid = record.id.split("|")[1] if "|" in record.id else record.id
    valid_proteins.add(pid)

# Load and filter terms
terms_df = pd.read_csv(TRAIN_TERMS_PATH, sep="\t", header=None, names=["pid", "term", "aspect"])
filtered_terms = terms_df[terms_df['pid'].isin(valid_proteins)]
raw_terms = set(filtered_terms['term'].unique())

# Load Graph and Expand Ancestors
go_graph = obonet.read_obo(OBO_PATH)
all_terms = raw_terms.copy()
print("   Expanding ancestors from OBO graph...")
for term in tqdm(raw_terms):
    if term in go_graph: all_terms.update(nx.ancestors(go_graph, term))

# Create Index Map (Sorted to match model output)
all_go_terms = sorted(list(all_terms))
GO_ID_TO_INDEX = {go: i for i, go in enumerate(all_go_terms)}
VOCAB_SIZE = len(GO_ID_TO_INDEX)
print(f"✅ Vocabulary Size: {VOCAB_SIZE}")

# --- 3. PREPARE EVALUATION ARTIFACTS ---
print("🗺️ Building Evaluation Maps & Weights...")
masks = {
    "BP": np.zeros(VOCAB_SIZE, dtype=bool),
    "MF": np.zeros(VOCAB_SIZE, dtype=bool),
    "CC": np.zeros(VOCAB_SIZE, dtype=bool)
}
weights = np.zeros(VOCAB_SIZE, dtype=np.float32)
ancestor_map = {}

# Get term counts for IA (Information Accretion) Weights
term_counts = filtered_terms['term'].value_counts().to_dict()
total_proteins = filtered_terms['pid'].nunique()

for term, idx in tqdm(GO_ID_TO_INDEX.items()):
    # 1. IA Weights: -log2(P(t))
    freq = (term_counts.get(term, 0) + 1) / (total_proteins + 1)
    weights[idx] = -np.log2(freq)
    
    if term in go_graph:
        # 2. Ancestor Map (for Propagation)
        ancs = {GO_ID_TO_INDEX[a] for a in nx.ancestors(go_graph, term) if a in GO_ID_TO_INDEX}
        if ancs: ancestor_map[idx] = ancs
        
        # 3. Ontology Masks
        ns = go_graph.nodes[term].get('namespace')
        if ns == 'biological_process': masks["BP"][idx] = True
        elif ns == 'molecular_function': masks["MF"][idx] = True
        elif ns == 'cellular_component': masks["CC"][idx] = True

# --- 4. DATASET & MODEL CLASS ---
class PTEmbeddingDataset(Dataset):
    def __init__(self, pt_dir, label_to_idx, num_classes):
        self.pt_files = sorted(glob.glob(os.path.join(pt_dir, "*.pt")))
        self.label_to_idx = label_to_idx
        self.num_classes = num_classes
    def __len__(self): return len(self.pt_files)
    def __getitem__(self, idx):
        data = torch.load(self.pt_files[idx], weights_only=True)
        emb = data["embedding"].float()
        target_tensor = torch.zeros(self.num_classes, dtype=torch.float32)
        
        # Handle labels (ensure flat list of strings)
        raw_labels = data.get("targets", [])
        if isinstance(raw_labels, list) and len(raw_labels)>0 and isinstance(raw_labels[0], list):
            raw_labels = [item for sublist in raw_labels for item in sublist]
            
        indices = [self.label_to_idx[t] for t in raw_labels if t in self.label_to_idx]
        if indices: target_tensor[list(set(indices))] = 1.0
        return emb, target_tensor

class EmbeddingClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.model(x)

# Setup Data (90% Train / 10% Validation)
full_ds = PTEmbeddingDataset(PT_DIR, GO_ID_TO_INDEX, VOCAB_SIZE)
train_len = int(0.9 * len(full_ds))
val_len = len(full_ds) - train_len
_, val_ds = random_split(full_ds, [train_len, val_len], generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

# --- 5. LOAD BEST MODEL ---
print("\n⚙️ Loading Model...")
model = EmbeddingClassifier(1280, VOCAB_SIZE).to(device)
model_path = "/kaggle/input/protien650esm/best_model_hierarchy.pth"

if os.path.exists(model_path):
    print(f"✅ Loading BEST saved weights from: {model_path}")
    model.load_state_dict(torch.load(model_path, map_location=device))
else:
    print("⚠️ WARNING: 'best_model_hierarchy.pth' not found. Using random initialized weights.")

# --- 6. RUN INFERENCE ---
print("\n🚀 Running Inference on Validation Set...")
model.eval()
val_probs = []
val_targets = []

with torch.no_grad():
    for embs, targets in tqdm(val_loader, desc="Inference"):
        embs = embs.to(device)
        out = model(embs)
        val_probs.append(torch.sigmoid(out).cpu().numpy())
        val_targets.append(targets.numpy())

y_prob_raw = np.vstack(val_probs)
y_true = np.vstack(val_targets)
print(f"   Inference Done. Predictions Shape: {y_prob_raw.shape}")

# Clean up memory
del model, val_loader, full_ds, val_ds
gc.collect()

# --- 7. PROPAGATE & EVALUATE (MICRO-AVERAGE) ---
print("\n🌊 Propagating Predictions (Enforcing Hierarchy)...")
# 1. Ancestor Propagation: Child Probability <= Parent Probability
# We use in-place updates for efficiency. 
# It ensures that if a specific term is predicted, its general parent is also predicted.
for child, parents in tqdm(ancestor_map.items(), desc="Propagating"):
    if parents:
        # Update all parents to be at least as confident as the child
        # Rule: Parent_Prob = max(Parent_Prob, Child_Prob)
        np.maximum(y_prob_raw[:, list(parents)], y_prob_raw[:, child][:, None], out=y_prob_raw[:, list(parents)])

print("\n📊 Calculating Micro-Averaged Metrics...")

def calculate_micro_metrics(y_true, y_prob, thresholds, mask=None):
    """
    Calculates Micro-Average F1, Precision, and Recall.
    Micro-average treats every sample-class pair as a single unit, 
    aggregating total True Positives, False Positives, and False Negatives globally.
    """
    # 1. Filter by Ontology Mask if provided (e.g., only BP terms)
    if mask is not None:
        yt = y_true[:, mask]
        yp = y_prob[:, mask]
    else:
        yt = y_true
        yp = y_prob

    # 2. Pre-calculate Total True Positives (Denominator for Recall)
    # Summing the entire matrix gives the total number of '1's in ground truth
    total_true_labels = yt.sum()
    
    best = {"F1": 0.0, "Precision": 0.0, "Recall": 0.0, "Threshold": 0.0}
    
    # 3. Iterate over thresholds to find the optimal one
    for t in thresholds:
        # Binarize predictions based on threshold
        pred_bin = (yp >= t).astype(int)
        
        # Calculate Micro Stats
        tp = (pred_bin * yt).sum()
        pred_sum = pred_bin.sum()  # Total Predicted Positives (TP + FP)
        
        # Calculate Metrics
        # Avoid division by zero
        precision = tp / pred_sum if pred_sum > 0 else 0.0
        recall = tp / total_true_labels if total_true_labels > 0 else 0.0
        
        if (precision + recall) > 0:
            f1 = (2 * precision * recall) / (precision + recall)
        else:
            f1 = 0.0
            
        # Update Best Score
        if f1 > best["F1"]:
            best = {
                "F1": f1, 
                "Precision": precision, 
                "Recall": recall, 
                "Threshold": t
            }
            
    return best

# --- Run the Evaluation Loop ---
results = []
thresholds = np.arange(0.1, 0.9, 0.05)

# Define the Ontologies to evaluate
ontologies = {
    "Global": None,
    "Biological Process (BP)": masks["BP"],
    "Molecular Function (MF)": masks["MF"],
    "Cellular Component (CC)": masks["CC"]
}

for name, mask in ontologies.items():
    print(f"   Evaluating {name}...")
    
    # Calculate Micro Metrics using the Propagated Probabilities
    m = calculate_micro_metrics(y_true, y_prob_raw, thresholds, mask)
    
    results.append({
        "Ontology": name,
        "Micro F1": f"{m['F1']:.4f}",
        "Micro Precision": f"{m['Precision']:.4f}",
        "Micro Recall": f"{m['Recall']:.4f}",
        "Best Threshold": f"{m['Threshold']:.2f}"
    })

# --- Display Results ---
print("\n🏆 FINAL MICRO-AVERAGED RESULTS (With Propagation) 🏆")
df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))

# Optional: Print simple LaTeX table for paper
print("\n-- LaTeX Table Body --")
for _, row in df_results.iterrows():
    print(f"{row['Ontology']} & {row['Micro F1']} & {row['Micro Precision']} & {row['Micro Recall']} & {row['Best Threshold']} \\\\")

⚙️ Setting up paths and device...
🔄 Rebuilding Vocabulary (GO_ID_TO_INDEX)...
   Expanding ancestors from OBO graph...


  0%|          | 0/26125 [00:00<?, ?it/s]

✅ Vocabulary Size: 39791
🗺️ Building Evaluation Maps & Weights...


  0%|          | 0/39791 [00:00<?, ?it/s]


⚙️ Loading Model...
✅ Loading BEST saved weights from: /kaggle/input/protien650esm/best_model_hierarchy.pth

🚀 Running Inference on Validation Set...


Inference:   0%|          | 0/129 [00:00<?, ?it/s]

   Inference Done. Predictions Shape: (8241, 39791)

🌊 Propagating Predictions (Enforcing Hierarchy)...


Propagating:   0%|          | 0/18006 [00:00<?, ?it/s]


📊 Calculating Micro-Averaged Metrics...
   Evaluating Global...
   Evaluating Biological Process (BP)...
   Evaluating Molecular Function (MF)...
   Evaluating Cellular Component (CC)...

🏆 FINAL MICRO-AVERAGED RESULTS (With Propagation) 🏆
| Ontology                |   Micro F1 |   Micro Precision |   Micro Recall |   Best Threshold |
|:------------------------|-----------:|------------------:|---------------:|-----------------:|
| Global                  |     0.5142 |            0.4716 |         0.5653 |              0.3 |
| Biological Process (BP) |     0.1864 |            0.1576 |         0.2281 |              0.1 |
| Molecular Function (MF) |     0.6483 |            0.5363 |         0.8193 |              0.3 |
| Cellular Component (CC) |     0.4528 |            0.3975 |         0.526  |              0.3 |

-- LaTeX Table Body --
Global & 0.5142 & 0.4716 & 0.5653 & 0.30 \\
Biological Process (BP) & 0.1864 & 0.1576 & 0.2281 & 0.10 \\
Molecular Function (MF) & 0.6483 & 0.5363 & 0.81